## Complex Number

### Complex Number

`z = a + bi`

### Complex arithmetic

#### Addition

`(a + bi) + (c + di) = (a + c) + (b + d)i`

#### Multiplication

`(a + bi)(c + di) = (ac - bd) + (ad + bc)i`

#### Conjugate

`conjugate of (a + bi) = a - bi`

#### Division

`(a + bi) / (c + di) = (a + bi)(c - di) / ((c + di)(c - di))`

### Polar form

`z = r * (cos(theta) + i* sin(theta))`

with:

`r = |z| = sqrt(a^2 + b^2)`

`theta = atan2(b, a)`



##### Eular's formula

`e^(i*theta) = cos(theta) + i*sin(theta)`

#### Multiplication

`z1 = r1 * e^(i * theta1)`

`z2 = r2 * e^(i * theta2)`

`z1 * z2 = (r1 * r2) * e^(i * (theta1 + theta2))`

**Eular's formula indicates that complex exponentials ARE rotations!!!**

```
(x + yi) * (cos(theta) + i * sin(theta)) = 
    (x * cos(theta) - y * sin(theta)) + 
    (x * sin(theta) + y * cos(theta)) * i
```

Similar to 2D Rotation
```
[cos(theta)  -sin(theta)] [x]   [x * cos(theta) - sin(theta) * y]
[sin(theta)   cos(theta)] [y] = [x * sin(theta) + sin(theta) * y]
```


### Connection to transformers

前面说了：**$e^{i\theta}$ = 旋转**。Transformer 里的位置信息，很多就是在用「按位置转一转」。

---

#### Sinusoidal positional encodings

原版 Transformer 给每个位置 `pos`、每个维度对 `(2i, 2i+1)` 加一组 sin/cos：

```
PE(pos, 2i)     = sin(pos / 10000^(2i / d))
PE(pos, 2i + 1) = cos(pos / 10000^(2i / d))
```

和欧拉公式对照：令 $\omega_i = 1 / 10000^{2i/d}$，则

```
(PE(pos, 2i+1), PE(pos, 2i))  ↔  Re/Im of  e^(i * pos * ω_i)
```

也就是：**每一对维度是单位圆上、角度随位置线性增加的一个点**。

为什么这样设计：

* 不同频率 $\omega_i$（不同 $i$）→ 长短距离都能区分
* 位置平移时，sin/cos 有线性组合关系 → 模型较容易学到 **相对位置**（$pos+k$ 相对 $pos$）
* 实现上通常直接写 sin/cos，不必真的建 `complex` 类型；数学上就是 $e^{i\theta}$

---

#### RoPE (Rotary Position Embedding)

RoPE 不把 PE **加**到向量上，而是按位置 **旋转** query / key 的二维子空间。

把相邻两维当成复数（或 2D 向量），对位置 $m$：

```
q_m' = q * e^(i * m * θ)
k_n' = k * e^(i * n * θ)
```

（对每对维度用各自的 $\theta_j$，类似正弦编码里不同频率。）

注意力里出现的是内积。旋转后：

```
<q_m', k_n'>  只依赖于 (m - n)
```

因为

```
(q e^(i m θ)) · (k e^(i n θ))  ∝  Re( q conj(k) * e^(i (m-n) θ) )
```

角度差 $m-n$ 才进公式 → **相对位置**直接编进注意力，绝对位置 $m$、$n$ 本身不单独留下。

和笔记上一节的对应：

| 复数 / 旋转 | Transformer |
|-------------|-------------|
| $z \cdot e^{i\theta}$ 旋转 $z$ | RoPE 按 token 位置旋转 $q,k$ |
| 两复数相乘 → 角度相加 | 两位置旋转后内积 → 只留角度差 |
| 极坐标 $r e^{i\theta}$ | 向量模长大致保留，方向编码位置 |

一句话：

* **Sinusoidal PE**：用 $e^{i\cdot pos\cdot\omega}$ 的实虚部当位置特征，**加**到 embedding 上
* **RoPE**：用 $e^{i\cdot pos\cdot\theta}$ **乘（旋转）** $q/k$，让注意力自然依赖相对距离

两者都建立在同一件事上：**复指数 = 旋转**。
